# 05 - KQL Real-Time Queries
**Eventhouse / KQL Database Queries for Real-Time Dashboards**

This notebook contains KQL (Kusto Query Language) queries designed to run against
the `bikerentaldb` KQL database in the Eventhouse. These power:

1. **Real-Time Dashboard** -- Live station availability map
2. **Anomaly Detection** -- Unusual availability patterns
3. **Fleet Rebalancing Alerts** -- Stations hitting critical levels
4. **Demand Heatmap** -- Neighbourhood-level demand intensity
5. **SLA Monitoring** -- Station uptime and availability SLAs

## How to Use
- Copy these KQL queries into the Eventhouse query editor
- Or use them in a Fabric Real-Time Dashboard
- Or reference them from Activator alert rules

> **Note**: The Python cells below contain the KQL as strings for reference.
> Run them against the Eventhouse using the KQL magic command or paste into the KQL editor.

In [ ]:
# ============================================================
# CELL 1: KQL Query Collection
# ============================================================
# These KQL queries are designed for the bikerentaldb Eventhouse.
# They assume a table called 'bikerentaldb' is being ingested
# via Eventstream.

kql_queries = {}

# ── 1. Real-Time Station Availability (last 5 minutes) ──
kql_queries["live_station_status"] = """
// Live Station Status — Last 5 minutes
// Shows current state of all stations with availability classification
// Uses ingestion_time() since raw stream has no timestamp column
bikerentaldb
| where ingestion_time() > ago(5m)
| summarize arg_max(ingestion_time(), No_Bikes, No_Empty_Docks) by BikepointID, Street, Neighbourhood, Latitude, Longitude
| extend total_docks = No_Bikes + No_Empty_Docks
| extend availability_pct = iff(total_docks == 0, 0.0, round(todouble(No_Bikes) / todouble(total_docks) * 100, 1))
| extend status = case(
    No_Bikes == 0, "EMPTY",
    No_Empty_Docks == 0, "FULL",
    availability_pct < 20, "LOW",
    availability_pct > 80, "HIGH",
    "NORMAL"
  )
| project BikepointID, Street, Neighbourhood,
    No_Bikes, No_Empty_Docks, total_docks,
    availability_pct, status,
    Latitude, Longitude
| order by availability_pct asc
| take 200
"""

print(kql_queries["live_station_status"])
print("1. live_station_status -- Real-time station availability")

In [ ]:
# ── 2. Fleet Rebalancing Alert Query ──
kql_queries["rebalance_alerts"] = """
// Fleet Rebalancing Alerts — Dispatch List
// Stations needing immediate attention (empty, full, or nearly so)
// Guard: iff() prevents division by zero on total_docks
bikerentaldb
| where ingestion_time() > ago(10m)
| summarize arg_max(ingestion_time(), No_Bikes, No_Empty_Docks) by BikepointID, Street, Neighbourhood, Latitude, Longitude
| extend total_docks = No_Bikes + No_Empty_Docks
| where No_Bikes <= 2 or No_Empty_Docks <= 2
| extend urgency = case(
    No_Bikes == 0, "CRITICAL - No bikes available",
    No_Empty_Docks == 0, "CRITICAL - Station full",
    No_Bikes <= 2, "WARNING - Very low bikes",
    No_Empty_Docks <= 2, "WARNING - Almost full",
    "MONITOR"
  )
| extend target_bikes = iff(total_docks == 0, 0, toint(round(todouble(total_docks) * 0.5)))
| extend bikes_needed = target_bikes - No_Bikes
| project BikepointID, Street, Neighbourhood,
    No_Bikes, No_Empty_Docks, total_docks,
    urgency, bikes_needed,
    Latitude, Longitude
| order by No_Bikes asc, No_Empty_Docks asc

| take 50
"""

print("2. rebalance_alerts -- Fleet rebalancing triggers")

print(kql_queries["rebalance_alerts"])

In [ ]:
# ── 3. Hourly Demand Trend (Rolling 24 Hours) ──
kql_queries["hourly_demand_trend"] = """
// Hourly Demand Trend — Last 24 Hours
// Aggregated demand patterns by hour and neighbourhood
// Computes utilization and critical flags inline from raw columns
bikerentaldb
| where ingestion_time() > ago(24h)
| extend total_docks = No_Bikes + No_Empty_Docks
| extend utilization_pct = iff(total_docks == 0, 0.0, todouble(No_Bikes) / todouble(total_docks))
| extend is_critical = No_Bikes <= 2 or No_Empty_Docks <= 2
| summarize
    event_count = count(),
    avg_utilization = round(avg(utilization_pct) * 100, 1),
    critical_events = countif(is_critical),
    empty_events = countif(No_Bikes == 0),
    full_events = countif(No_Empty_Docks == 0),
    unique_stations = dcount(BikepointID)
    by bin(ingestion_time(), 1h), Neighbourhood
| extend hour_of_day = datetime_part("Hour", ingestion_time_)
| extend is_rush = hour_of_day between (7 .. 9) or hour_of_day between (17 .. 19)
| order by ingestion_time_ asc, Neighbourhood asc
"""

print(kql_queries["hourly_demand_trend"])
print("3. hourly_demand_trend -- 24-hour rolling demand")

In [ ]:
# ── 4. Neighbourhood Demand Heatmap ──
kql_queries["neighbourhood_heatmap"] = """
// Neighbourhood Demand Heatmap — Last 1 Hour
// Aggregated demand intensity by neighbourhood
// All metrics computed inline from raw columns
bikerentaldb
| where ingestion_time() > ago(1h)
| extend total_docks = No_Bikes + No_Empty_Docks
| extend utilization_pct = iff(total_docks == 0, 0.0, todouble(No_Bikes) / todouble(total_docks))
| extend is_critical = No_Bikes <= 2 or No_Empty_Docks <= 2
| summarize
    total_events = count(),
    avg_bikes = round(avg(No_Bikes), 1),
    avg_utilization = round(avg(utilization_pct) * 100, 1),
    critical_count = countif(is_critical),
    station_count = dcount(BikepointID),
    avg_lat = avg(Latitude),
    avg_lon = avg(Longitude)
    by Neighbourhood
| extend demand_level = case(
    total_events > 100, "VERY HIGH",
    total_events > 50, "HIGH",
    total_events > 20, "MEDIUM",
    "LOW"
  )
| extend health_indicator = case(
    critical_count > 5, "RED",
    critical_count > 2, "YELLOW",
    "GREEN"
  )
| order by critical_count desc
"""

print("4. neighbourhood_heatmap -- Demand intensity map")
print(kql_queries["neighbourhood_heatmap"])

In [ ]:
# ── 5. Anomaly Detection (Unusual Patterns) ──
kql_queries["anomaly_detection"] = """
// Anomaly Detection — Stations with Unusual Activity
// Z-score comparison: recent 30min vs 24h baseline
// Uses iff() to guard against zero standard deviation
let baseline = bikerentaldb
| where ingestion_time() between (ago(24h) .. ago(1h))
| summarize
    baseline_avg_bikes = avg(No_Bikes),
    baseline_std = stdev(No_Bikes)
    by BikepointID;
let recent = bikerentaldb
| where ingestion_time() > ago(30m)
| summarize
    recent_avg_bikes = avg(No_Bikes),
    recent_events = count(),
    latest_update = max(ingestion_time())
    by BikepointID, Street, Neighbourhood;
recent
| join kind=inner baseline on BikepointID
| extend z_score = round(
    (recent_avg_bikes - baseline_avg_bikes) / iff(baseline_std < 0.01, 1.0, baseline_std), 2)
| where abs(z_score) > 2.0
| extend anomaly_type = case(
    z_score > 2, "SURGE - Unusual bike accumulation",
    z_score < -2, "DRAIN - Unusual bike depletion",
    "NORMAL"
  )
| project BikepointID, Street, Neighbourhood,
    recent_avg_bikes = round(recent_avg_bikes, 1),
    baseline_avg_bikes = round(baseline_avg_bikes, 1),
    z_score, anomaly_type, recent_events, latest_update
| order by abs(z_score) desc
| take 50
"""

print("5. anomaly_detection -- Z-score based anomaly detection")
print(kql_queries["anomaly_detection"])

In [ ]:
# ── 6. SLA Monitoring ──
kql_queries["sla_monitoring"] = """
// SLA Monitoring — Station Availability Compliance
// Tracks % of check-ins where station had bikes available
// SLA Target: 95% availability (bikes > 0)
bikerentaldb
| where ingestion_time() > ago(24h)
| summarize
    total_checks = count(),
    available_checks = countif(No_Bikes > 0),
    empty_checks = countif(No_Bikes == 0),
    avg_bikes = round(avg(No_Bikes), 1),
    min_bikes = min(No_Bikes),
    first_seen = min(ingestion_time()),
    last_seen = max(ingestion_time())
    by BikepointID, Street, Neighbourhood
| extend sla_pct = iff(total_checks == 0, 0.0, round(todouble(available_checks) / todouble(total_checks) * 100, 2))
| extend sla_status = case(
    sla_pct >= 95, "COMPLIANT",
    sla_pct >= 80, "AT RISK",
    "BREACHED"
  )
| extend monitoring_hours = round(datetime_diff("minute", last_seen, first_seen) / 60.0, 1)
| project BikepointID, Street, Neighbourhood,
    sla_pct, sla_status, total_checks,
    available_checks, empty_checks, avg_bikes,
    monitoring_hours
| order by sla_pct asc
| take 200
"""

print("6. sla_monitoring -- Availability SLA compliance")
print(kql_queries["sla_monitoring"])

In [ ]:
# ── 7. Dynamic Pricing Signals ──
kql_queries["pricing_signals"] = """
// Dynamic Pricing Signals
// Generate price multipliers based on supply-demand imbalance
// Guard: iff() prevents division by zero when total_docks = 0
bikerentaldb
| where ingestion_time() > ago(15m)
| summarize arg_max(ingestion_time(), No_Bikes, No_Empty_Docks) by BikepointID, Street, Neighbourhood, Latitude, Longitude
| extend total_docks = No_Bikes + No_Empty_Docks
| extend supply_ratio = iff(total_docks == 0, 0.0, round(todouble(No_Bikes) / todouble(total_docks), 3))
| extend price_multiplier = case(
    supply_ratio < 0.1, 2.0,
    supply_ratio < 0.2, 1.5,
    supply_ratio > 0.9, 0.5,
    supply_ratio > 0.8, 0.75,
    1.0
  )
| extend pricing_reason = case(
    price_multiplier > 1.0, strcat("Supply shortage (", tostring(round(supply_ratio * 100, 0)), "%)"),
    price_multiplier < 1.0, strcat("Oversupply discount (", tostring(round(supply_ratio * 100, 0)), "%)"),
    "Standard rate"
  )
| project BikepointID, Street, Neighbourhood,
    No_Bikes, No_Empty_Docks, total_docks,
    supply_ratio, price_multiplier, pricing_reason,
    Latitude, Longitude
| order by price_multiplier desc
| take 200
"""

print(kql_queries["pricing_signals"])
print("7. pricing_signals -- Dynamic pricing based on supply/demand")

In [ ]:
# ── 8. Time-Series Trend (for Grafana / RT Dashboard) ──
kql_queries["timeseries_trend"] = """
// Time-Series Availability Trend — 5min Resolution
// For real-time dashboard line charts
// Computes all metrics inline from raw columns
bikerentaldb
| where ingestion_time() > ago(6h)
| extend total_docks = No_Bikes + No_Empty_Docks
| extend utilization_pct = iff(total_docks == 0, 0.0, todouble(No_Bikes) / todouble(total_docks))
| extend is_critical = No_Bikes <= 2 or No_Empty_Docks <= 2
| summarize
    avg_bikes = round(avg(No_Bikes), 1),
    avg_utilization = round(avg(utilization_pct) * 100, 1),
    total_events = count(),
    critical_events = countif(is_critical),
    stations_reporting = dcount(BikepointID)
    by bin(ingestion_time(), 5m)
| order by ingestion_time_ asc
| render timechart
"""

print("8. timeseries_trend -- 5-min resolution for dashboards")
print(kql_queries["timeseries_trend"])

In [ ]:
# ============================================================
# CELL 9: Export KQL Queries to Files
# ============================================================
# Save KQL queries as .kql files for easy import into Eventhouse

import os

# Print summary
print("=" * 60)
print("KQL QUERY COLLECTION")
print("=" * 60)
for name, query in kql_queries.items():
    lines = [l for l in query.strip().split('\n') if l.strip() and not l.strip().startswith('//')]
    print(f"  {name:<30} ({len(lines)} KQL lines)")

print(f"\nTotal queries: {len(kql_queries)}")

# Save queries for easy access
kql_output_dir = "/lakehouse/default/Files/kql_queries"
os.makedirs(kql_output_dir, exist_ok=True)
for name, query in kql_queries.items():
    with open(f"{kql_output_dir}/{name}.kql", "w") as f:
        f.write(query.strip())
    print(f"  Saved: {kql_output_dir}/{name}.kql")


print("\nTo use these queries:")
print("  1. Open bikerentaleventhouse in Fabric")
print("  2. Open the KQL query editor")
print("  3. Paste any query above")
print("  4. Or create a Real-Time Dashboard with these as tiles")
print("\nFor Activator alerts, use:")
print("  - rebalance_alerts (triggers fleet dispatch)")
print("  - sla_monitoring (triggers SLA breach notification)")
print("  - anomaly_detection (triggers investigation)")